In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from Module.panel_utils import (
    ModelResultsAggregator,
    run_panel_regressions,
    run_spec_tests,
    run_panel_model_diagnostics,
)
s

In [54]:
###############################
# ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
###############################
df_reg_analys = pd.read_excel('reg_analys.xlsx')
df_fed_analys = pd.read_excel('fed_analys.xlsx')


# Убираем служебные столбцы индекса
df_reg_analys = df_reg_analys.loc[:, ~df_reg_analys.columns.str.startswith('Unnamed')]
df_fed_analys = df_fed_analys.loc[:, ~df_fed_analys.columns.str.startswith('Unnamed')]

# Приводим даты к datetime
df_reg_analys['Date'] = pd.to_datetime(df_reg_analys['Date'])
df_fed_analys['Date'] = pd.to_datetime(df_fed_analys['Date'])

# Разделим d_Mon_Shock на позитивный и негативный
df_reg_analys['d_Mon_Shock_neg'] = df_reg_analys['d_Mon_Shock'].where(df_reg_analys['d_Mon_Shock'] < 0, 0)
df_reg_analys['d_Mon_Shock_pos'] = df_reg_analys['d_Mon_Shock'].where(df_reg_analys['d_Mon_Shock'] > 0, 0)

# Объединяем региональные и федеральные данные
fed_extra_cols = [
    col for col in df_fed_analys.columns
    if col not in df_reg_analys.columns and col != 'Region'
]
df_reg = df_reg_analys.merge(
    df_fed_analys[['Date'] + fed_extra_cols],
    on='Date',
    how='left'
)


# Взаимодействия (если требуется)
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_1' in df_reg.columns:
    df_reg['d_Mon_Shock_Cl1'] = df_reg['d_Mon_Shock'] * df_reg['Cluster_1']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg['d_Mon_Shock_Cl2'] = df_reg['d_Mon_Shock'] * df_reg['Cluster_2']
if 'd_Mon_Shock' in df_reg.columns and 'Covid_dum' in df_reg.columns:
    df_reg['d_Mon_Shock_Covid'] = df_reg['d_Mon_Shock'] * df_reg['Covid_dum']

# Кластерные выборки
if 'Cluster_1' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg_clus_one = df_reg[df_reg['Cluster_1'] == 1].copy()
    df_reg_clus_two = df_reg[df_reg['Cluster_2'] == 1].copy()
    df_reg_clus_three = df_reg[(df_reg['Cluster_1'] == 0) & (df_reg['Cluster_2'] == 0)].copy()


In [55]:
df_reg.columns

Index(['Region', 'Date', 'Int_Rate_FL', 'Int_Rate_FL_lag1', 'Int_Rate_Mort',
       'Credit_impulse', 'Int_Rate_Mort_lag1', 'Int_Rate_ConsCred',
       'Int_Rate_ConsCred_lag1', 'Cred_nagr', 'D_top5_rozn', 'Fin_Dostup',
       'Cred_structure', 'Def_Zadolg_Fl', 'Def_Zadolg_Mort',
       'Def_Zadolg_ConsCred', 'Mon_Shock', 'Exc_rate',
       'Inflation_Expectations', 'Bonds_Rate_Correct_5Y', 'Cluster_1',
       'Cluster_2', 'New_Loans_Progr', 'MIACR', 'Covid_dum', 'Sank_dum',
       'ROISFIX', 'Inflation_Expectations_adj', 'Inflation_Expectations_trend',
       'Inflation_Expectations_seasonal', 'd_Int_Rate_FL',
       'd_Int_Rate_ConsCred', 'Mon_Shock_neg', 'Mon_Shock_pos', 'd_Mon_Shock',
       'd_Mon_Shock_neg', 'd_Mon_Shock_pos', 'd_Inflation_Expectations',
       'Num_reg', 'CAR_Indicator', 'D_Progr_mort', 'exc_rate', 'exc_rate_diff',
       'Nominal_Percent_Rate', 'Ent_conf_ind_mining',
       'Ent_conf_ind_manufactoring', 'Key_Rate',
       'Ent_conf_ind_manufactoring_adj', 'Ent_

In [56]:
###############################
# Построение линейных моделей на панельных данных (общая выборка)
# (Int_Rate_Mort зависимая переменная)
###############################


exog_vars_initial = [
    'Int_Rate_Mort_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Mort',
    'ln_New_Loans_Mort',
    'd_Mon_Shock',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations',
    'Covid_dum',
    'Sank_dum'
]

dependent_var = 'Int_Rate_Mort'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)
Зависимая переменная: Int_Rate_Mort

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:          Int_Rate_Mort   R-squared:                        0.9941
Estimator:                  PooledOLS   R-squared (Between):              0.9998
No. Observations:                6237   R-squared (Within):               0.7221
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9941
Time:                        15:24:33   Log-likelihood                   -6051.6
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   8.093e+04
Entities:                          81   P-value                           0.0000
Avg Obs:                       77.000   Distribution:                 F(13,6224)
Min Obs:                       77.000                                           

In [57]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_Mort
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable     t_stat    p_val
      Int_Rate_Mort_lag1  -9.355636 0.000000
         Def_Zadolg_Mort  -8.435292 0.000000
               Covid_dum -11.543072 0.000000
          Cred_structure  -7.758717 0.000000
   Bonds_Rate_Correct_5Y  -7.342012 0.000000
             d_Mon_Shock  -6.921762 0.000000
               d_Ex_Rate  -6.906433 0.000000
d_Inflation_Expectations   6.351084 0.000000
             D_top5_rozn  -6.269640 0.000000
          Credit_impulse   5.936565 0.000000
               Cred_nagr  -5.880604 0.000000
                Sank_dum   5.017529 0.000001
              Fin_Dostup  -3.113121 0.001860

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Br

In [58]:
# Сохраняем результаты модели для экспорта
pooled_res_1 = pooled_res if pooled_success else None
fe_res_1 = fe_res if fe_success else None
re_res_1 = re_res if re_success else None


In [59]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: -39.7718
p-значение: 1.000000
df: 13

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 2979.5690
p-значение: 0.000000
N (регионов): 81, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -71.9657
p-значение: 1.000000
df: (80, 6143)

Вывод: p >= 0.05 - Pooled адекватна


(-39.771775710225995, 1.0, 2979.5690022029667, 0.0, -71.96568044677672, 1.0)

In [60]:
###############################
# Построение линейных моделей на панельных данных (общая выборка)
# (Int_Rate_Mort зависимая переменная)
###############################


exog_vars_initial = [
    'Int_Rate_Mort_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Mort',
    'ln_New_Loans_Mort',
    'd_Mon_Shock',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations',
    'Covid_dum',
    'Sank_dum',
    'd_Mon_Shock_Covid'
]

dependent_var = 'Int_Rate_Mort'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)
Зависимая переменная: Int_Rate_Mort

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:          Int_Rate_Mort   R-squared:                        0.9941
Estimator:                  PooledOLS   R-squared (Between):              0.9998
No. Observations:                6237   R-squared (Within):               0.7230
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9941
Time:                        15:24:35   Log-likelihood                   -6041.3
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   7.539e+04
Entities:                          81   P-value                           0.0000
Avg Obs:                       77.000   Distribution:                 F(14,6223)
Min Obs:                       77.000                                           

In [61]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_Mort
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable     t_stat    p_val
      Int_Rate_Mort_lag1  -9.439494 0.000000
         Def_Zadolg_Mort  -8.520545 0.000000
               Covid_dum -11.618273 0.000000
          Cred_structure  -7.823877 0.000000
       d_Mon_Shock_Covid   7.637656 0.000000
   Bonds_Rate_Correct_5Y  -7.434322 0.000000
             d_Mon_Shock  -7.187240 0.000000
               d_Ex_Rate  -7.023394 0.000000
d_Inflation_Expectations   6.421045 0.000000
             D_top5_rozn  -6.358195 0.000000
          Credit_impulse   6.037977 0.000000
               Cred_nagr  -5.979799 0.000000
                Sank_dum   5.102937 0.000000
              Fin_Dostup  -3.037327 0.002397

----------------------------------------------------------------------
MODEL: Pooled OLS
------------------------------------------------------

In [62]:
# Сохраняем результаты модели для экспорта
pooled_res_2 = pooled_res if pooled_success else None
fe_res_2 = fe_res if fe_success else None
re_res_2 = re_res if re_success else None


In [63]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: -120.5224
p-значение: 1.000000
df: 14

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 2979.7708
p-значение: 0.000000
N (регионов): 81, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -72.1908
p-значение: 1.000000
df: (80, 6142)

Вывод: p >= 0.05 - Pooled адекватна


(-120.52237391024869, 1.0, 2979.7708134062755, 0.0, -72.1908217227696, 1.0)

In [64]:
###############################
# Построение линейных моделей на панельных данных (общая выборка)
# (Int_Rate_Mort зависимая переменная)
###############################


exog_vars_initial = [
    'Int_Rate_Mort_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Mort',
    'ln_New_Loans_Mort',
    'd_Mon_Shock',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations',
    'Covid_dum',
    'Sank_dum',
    'd_Mon_Shock_Cl1',
    'd_Mon_Shock_Cl2'
]

dependent_var = 'Int_Rate_Mort'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)
Зависимая переменная: Int_Rate_Mort

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:          Int_Rate_Mort   R-squared:                        0.9941
Estimator:                  PooledOLS   R-squared (Between):              0.9998
No. Observations:                6237   R-squared (Within):               0.7228
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9941
Time:                        15:24:37   Log-likelihood                   -6044.3
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   7.029e+04
Entities:                          81   P-value                           0.0000
Avg Obs:                       77.000   Distribution:                 F(15,6222)
Min Obs:                       77.000                                           

In [65]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_Mort
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable     t_stat    p_val
      Int_Rate_Mort_lag1  -9.335013 0.000000
         Def_Zadolg_Mort  -8.418507 0.000000
               Covid_dum -11.518582 0.000000
          Cred_structure  -7.744257 0.000000
             d_Mon_Shock  -7.377860 0.000000
   Bonds_Rate_Correct_5Y  -7.325064 0.000000
               d_Ex_Rate  -6.890274 0.000000
d_Inflation_Expectations   6.335329 0.000000
             D_top5_rozn  -6.254495 0.000000
          Credit_impulse   5.921393 0.000000
               Cred_nagr  -5.865812 0.000000
         d_Mon_Shock_Cl2  -5.138106 0.000000
                Sank_dum   5.006208 0.000001
              Fin_Dostup  -3.114514 0.001851
         d_Mon_Shock_Cl1  -1.515314 0.129744

----------------------------------------------------------------------
MODEL: Pooled OLS
---------

In [66]:
# Сохраняем результаты модели для экспорта
pooled_res_3 = pooled_res if pooled_success else None
fe_res_3 = fe_res if fe_success else None
re_res_3 = re_res if re_success else None


In [67]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 130.3822
p-значение: 0.000000
df: 15

Вывод: p < 0.05 - Используйте FIXED EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 2979.9509
p-значение: 0.000000
N (регионов): 81, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -71.9318
p-значение: 1.000000
df: (80, 6141)

Вывод: p >= 0.05 - Pooled адекватна


(130.3821930985171, 0.0, 2979.950926636565, 0.0, -71.93175118968789, 1.0)

In [68]:
###############################
# Построение линейных моделей на панельных данных (общая выборка)
# (Int_Rate_Mort зависимая переменная)
###############################


exog_vars_initial = [
    'Int_Rate_Mort_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Mort',
    'ln_New_Loans_Mort',
    'd_Mon_Shock',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations',
    'ln_New_Loans_Progr'
]

dependent_var = 'Int_Rate_Mort'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)
Зависимая переменная: Int_Rate_Mort

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:          Int_Rate_Mort   R-squared:                        0.9941
Estimator:                  PooledOLS   R-squared (Between):              0.9999
No. Observations:                6237   R-squared (Within):               0.7180
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9941
Time:                        15:24:40   Log-likelihood                   -6080.0
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   9.481e+04
Entities:                          81   P-value                           0.0000
Avg Obs:                       77.000   Distribution:                 F(11,6226)
Min Obs:                       77.000                                           

In [69]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_Mort
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
              Fin_Dostup -9.520034 0.000000
      Int_Rate_Mort_lag1 -7.115303 0.000000
         Def_Zadolg_Mort -6.421945 0.000000
          Cred_structure -6.265364 0.000000
   Bonds_Rate_Correct_5Y -5.735361 0.000000
             d_Mon_Shock -5.665697 0.000000
               d_Ex_Rate -5.049661 0.000000
             D_top5_rozn -4.602475 0.000004
               Cred_nagr -4.342663 0.000014
d_Inflation_Expectations -4.294310 0.000018
          Credit_impulse  4.209592 0.000026

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=284.1243, p=0.000000
White:         stat=967.5632, p=0.000000

Normality tests
Jarqu

In [70]:
# Сохраняем результаты модели для экспорта
pooled_res_4 = pooled_res if pooled_success else None
fe_res_4 = fe_res if fe_success else None
re_res_4 = re_res if re_success else None


In [71]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 86.3417
p-значение: 0.000000
df: 11

Вывод: p < 0.05 - Используйте FIXED EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 3014.3431
p-значение: 0.000000
N (регионов): 81, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -70.0935
p-значение: 1.000000
df: (80, 6145)

Вывод: p >= 0.05 - Pooled адекватна


(86.34172854951822,
 8.64863736182997e-14,
 3014.343130106296,
 0.0,
 -70.09352072249585,
 1.0)

In [72]:
###############################
# Построение линейных моделей на панельных данных (общая выборка)
# (Int_Rate_Mort зависимая переменная)
###############################


exog_vars_initial = [
    'Int_Rate_Mort_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Mort',
    'ln_New_Loans_Mort',
    'd_Mon_Shock',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations',
    'ln_New_Loans_Progr',
    'd_Mon_Shock_Cl1',
    'd_Mon_Shock_Cl2'
]

dependent_var = 'Int_Rate_Mort'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)
Зависимая переменная: Int_Rate_Mort

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:          Int_Rate_Mort   R-squared:                        0.9941
Estimator:                  PooledOLS   R-squared (Between):              0.9999
No. Observations:                6237   R-squared (Within):               0.7186
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9941
Time:                        15:24:42   Log-likelihood                   -6072.6
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   8.039e+04
Entities:                          81   P-value                           0.0000
Avg Obs:                       77.000   Distribution:                 F(13,6224)
Min Obs:                       77.000                                           

In [73]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_Mort
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
              Fin_Dostup -9.500780 0.000000
      Int_Rate_Mort_lag1 -7.095661 0.000000
             d_Mon_Shock -6.469973 0.000000
         Def_Zadolg_Mort -6.405439 0.000000
          Cred_structure -6.250746 0.000000
   Bonds_Rate_Correct_5Y -5.718543 0.000000
               d_Ex_Rate -5.033739 0.000000
             D_top5_rozn -4.587871 0.000005
               Cred_nagr -4.328555 0.000015
d_Inflation_Expectations -4.298539 0.000017
          Credit_impulse  4.195267 0.000028
         d_Mon_Shock_Cl2 -2.913439 0.003587
         d_Mon_Shock_Cl1 -0.097444 0.922377

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: s

In [74]:
# Сохраняем результаты модели для экспорта
pooled_res_5 = pooled_res if pooled_success else None
fe_res_5 = fe_res if fe_success else None
re_res_5 = re_res if re_success else None


In [75]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 76.4578
p-значение: 0.000000
df: 13

Вывод: p < 0.05 - Используйте FIXED EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 3014.6395
p-значение: 0.000000
N (регионов): 81, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -70.0555
p-значение: 1.000000
df: (80, 6143)

Вывод: p >= 0.05 - Pooled адекватна


(76.45782632942823,
 5.089895172005754e-11,
 3014.63953547461,
 0.0,
 -70.05546942250746,
 1.0)

In [76]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_Mort
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
              Fin_Dostup -9.500780 0.000000
      Int_Rate_Mort_lag1 -7.095661 0.000000
             d_Mon_Shock -6.469973 0.000000
         Def_Zadolg_Mort -6.405439 0.000000
          Cred_structure -6.250746 0.000000
   Bonds_Rate_Correct_5Y -5.718543 0.000000
               d_Ex_Rate -5.033739 0.000000
             D_top5_rozn -4.587871 0.000005
               Cred_nagr -4.328555 0.000015
d_Inflation_Expectations -4.298539 0.000017
          Credit_impulse  4.195267 0.000028
         d_Mon_Shock_Cl2 -2.913439 0.003587
         d_Mon_Shock_Cl1 -0.097444 0.922377

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: s

In [77]:
# Сохраняем результаты модели для экспорта
pooled_res_6 = pooled_res if pooled_success else None
fe_res_6 = fe_res if fe_success else None
re_res_6 = re_res if re_success else None


In [78]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 76.4578
p-значение: 0.000000
df: 13

Вывод: p < 0.05 - Используйте FIXED EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 3014.6395
p-значение: 0.000000
N (регионов): 81, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -70.0555
p-значение: 1.000000
df: (80, 6143)

Вывод: p >= 0.05 - Pooled адекватна


(76.45782632942823,
 5.089895172005754e-11,
 3014.63953547461,
 0.0,
 -70.05546942250746,
 1.0)

In [79]:
###############################
# Построение линейных моделей на панельных данных (кластер 2)
# (Int_Rate_Mort зависимая переменная)
###############################

exog_vars_initial = [
    'Int_Rate_Mort_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Mort',
   # 'ln_New_Loans_Progr',
    'ln_New_Loans_Mort',
    'd_Mon_Shock',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations'                        
]

dependent_var = 'Int_Rate_Mort'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg_clus_two, df_reg, dependent_var, exog_vars_initial, cov_type='robust', cluster_entity=None, df_name='df_reg_clus_two'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 2)
Зависимая переменная: Int_Rate_Mort

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:          Int_Rate_Mort   R-squared:                        0.9878
Estimator:                  PooledOLS   R-squared (Between):              0.9997
No. Observations:                 462   R-squared (Within):               0.6304
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9878
Time:                        15:24:43   Log-likelihood                   -583.77
Cov. Estimator:                Robust                                           
                                        F-statistic:                      3332.7
Entities:                           6   P-value                           0.0000
Avg Obs:                       77.000   Distribution:                  F(11,451)
Min Obs:                       77.000                                           
Max

In [80]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_Mort
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
      Int_Rate_Mort_lag1 -1.855716 0.064148
          Cred_structure -1.757531 0.079507
              Fin_Dostup -1.522945 0.128475
   Bonds_Rate_Correct_5Y -1.433646 0.152368
d_Inflation_Expectations -1.297972 0.194962
             D_top5_rozn  1.147605 0.251742
         Def_Zadolg_Mort  1.015667 0.310334
             d_Mon_Shock -0.817677 0.413974
               d_Ex_Rate  0.684591 0.493955
               Cred_nagr  0.638187 0.523677
          Credit_impulse  0.048975 0.960961

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=88.4065, p=0.000000
White:         stat=237.1361, p=0.000000

Normality tests
Jarque

In [81]:
# Сохраняем результаты модели для экспорта
pooled_res_7 = pooled_res if pooled_success else None
fe_res_7 = fe_res if fe_success else None
re_res_7 = re_res if re_success else None


In [82]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 5.4764
p-значение: 0.905924
df: 11

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 224.6472
p-значение: 0.000000
N (регионов): 6, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -88.5842
p-значение: 1.000000
df: (5, 445)

Вывод: p >= 0.05 - Pooled адекватна


(5.47637113100304,
 0.9059238778040567,
 224.64720837269883,
 0.0,
 -88.58422764923651,
 1.0)

In [83]:
###############################
# Построение линейных моделей на панельных данных (кластер 3)
# (Int_Rate_Mort зависимая переменная)
###############################

exog_vars_initial = [
    'Int_Rate_Mort_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Mort',
   # 'ln_New_Loans_Progr',
    'ln_New_Loans_Mort',
    'd_Mon_Shock',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations'                        
]

dependent_var = 'Int_Rate_Mort'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg_clus_three, df_reg, dependent_var, exog_vars_initial, cov_type='robust', cluster_entity=None, df_name='df_reg_clus_three'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 3)
Зависимая переменная: Int_Rate_Mort

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:          Int_Rate_Mort   R-squared:                        0.9959
Estimator:                  PooledOLS   R-squared (Between):              1.0000
No. Observations:                1463   R-squared (Within):               0.7550
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9959
Time:                        15:24:44   Log-likelihood                   -1139.5
Cov. Estimator:                Robust                                           
                                        F-statistic:                   3.211e+04
Entities:                          19   P-value                           0.0000
Avg Obs:                       77.000   Distribution:                 F(11,1452)
Min Obs:                       77.000                                           
Max

In [84]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_Mort
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
              Fin_Dostup -2.865875 0.004219
      Int_Rate_Mort_lag1 -2.267865 0.023484
         Def_Zadolg_Mort -1.916961 0.055439
             d_Mon_Shock -1.879812 0.060334
          Credit_impulse  1.784010 0.074631
   Bonds_Rate_Correct_5Y -1.763213 0.078075
               d_Ex_Rate -1.752196 0.079951
               Cred_nagr -1.671400 0.094858
          Cred_structure -1.628884 0.103555
             D_top5_rozn -1.360048 0.174026
d_Inflation_Expectations  0.518253 0.604360

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=83.6929, p=0.000000
White:         stat=367.2640, p=0.000000

Normality tests
Jarque

In [85]:
# Сохраняем результаты модели для экспорта
pooled_res_8 = pooled_res if pooled_success else None
fe_res_8 = fe_res if fe_success else None
re_res_8 = re_res if re_success else None


In [86]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 19.7200
p-значение: 0.049332
df: 11

Вывод: p < 0.05 - Используйте FIXED EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 725.8683
p-значение: 0.000000
N (регионов): 19, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -72.0494
p-значение: 1.000000
df: (18, 1433)

Вывод: p >= 0.05 - Pooled адекватна


(19.71997340560896,
 0.0493321307801603,
 725.8683167886192,
 0.0,
 -72.04936472575517,
 1.0)

In [87]:
###########################################
 #ПОСТРОЕНИЕ МОДЕЛЕЙ С ROISFIX
###########################################
# df_exog = pd.read_excel('exog.xlsx')

In [88]:
###############################
# Построение линейных моделей на панельных данных (общая выборка) ROISFIX
# (Int_Rate_Mort зависимая переменная)
###############################


exog_vars_initial = [
    'Int_Rate_Mort_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Mort',
   # 'ln_New_Loans_Progr',
    'ln_New_Loans_Mort',
    'ROISFIX',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations'                        
]

dependent_var = 'Int_Rate_Mort'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА) - ROISFIX
Зависимая переменная: Int_Rate_Mort

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:          Int_Rate_Mort   R-squared:                        0.9941
Estimator:                  PooledOLS   R-squared (Between):              0.9999
No. Observations:                6317   R-squared (Within):               0.7230
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9941
Time:                        15:24:44   Log-likelihood                   -6170.3
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   9.625e+04
Entities:                          81   P-value                           0.0000
Avg Obs:                       77.988   Distribution:                 F(11,6306)
Min Obs:                       77.000                                 

In [89]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_Mort
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
d_Inflation_Expectations -8.394501 0.000000
      Int_Rate_Mort_lag1 -8.029411 0.000000
         Def_Zadolg_Mort -7.775043 0.000000
          Cred_structure -7.575754 0.000000
                 ROISFIX  6.755994 0.000000
          Credit_impulse  5.866974 0.000000
             D_top5_rozn -5.838683 0.000000
               d_Ex_Rate -5.676767 0.000000
               Cred_nagr -5.107261 0.000000
   Bonds_Rate_Correct_5Y -4.510668 0.000007
              Fin_Dostup -3.343613 0.000832

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=318.2101, p=0.000000
White:         stat=1136.9981, p=0.000000

Normality tests
Jarq

In [90]:
# Сохраняем результаты модели для экспорта
pooled_res_9 = pooled_res if pooled_success else None
fe_res_9 = fe_res if fe_success else None
re_res_9 = re_res if re_success else None


In [91]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: -33.6944
p-значение: 1.000000
df: 11

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 3023.6334
p-значение: 0.000000
N (регионов): 81, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -74.9617
p-значение: 1.000000
df: (80, 6226)

Вывод: p >= 0.05 - Pooled адекватна


(-33.694378826861424, 1.0, 3023.6334135576963, 0.0, -74.96165211016921, 1.0)

In [92]:
###############################
# Построение линейных моделей на панельных данных (кластер 1 ROISFIX)
# (Int_Rate_Mort зависимая переменная)
###############################

exog_vars_initial = [
    'Int_Rate_Mort_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Mort',
   # 'ln_New_Loans_Progr',
    'ln_New_Loans_Mort',
    'ROISFIX',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations'                        
]

dependent_var = 'Int_Rate_Mort'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg_clus_one, df_reg, dependent_var, exog_vars_initial, cov_type='robust', cluster_entity=None, df_name='df_reg_clus_one'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 1 ROISFIX)
Зависимая переменная: Int_Rate_Mort

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:          Int_Rate_Mort   R-squared:                        0.9941
Estimator:                  PooledOLS   R-squared (Between):              0.9999
No. Observations:                4367   R-squared (Within):               0.7282
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9941
Time:                        15:24:45   Log-likelihood                   -4289.6
Cov. Estimator:                Robust                                           
                                        F-statistic:                   6.711e+04
Entities:                          56   P-value                           0.0000
Avg Obs:                       77.982   Distribution:                 F(11,4356)
Min Obs:                       77.000                                       

In [93]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_Mort
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
              Fin_Dostup -7.715519 0.000000
d_Inflation_Expectations -7.420454 0.000000
         Def_Zadolg_Mort -7.266137 0.000000
      Int_Rate_Mort_lag1 -7.224254 0.000000
          Cred_structure -6.723162 0.000000
                 ROISFIX  6.527856 0.000000
             D_top5_rozn -5.470023 0.000000
               d_Ex_Rate -5.018522 0.000001
          Credit_impulse  4.854926 0.000001
               Cred_nagr -3.891030 0.000101
   Bonds_Rate_Correct_5Y -1.800154 0.071905

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=229.5143, p=0.000000
White:         stat=826.2731, p=0.000000

Normality tests
Jarqu

In [94]:
# Сохраняем результаты модели для экспорта
pooled_res_10 = pooled_res if pooled_success else None
fe_res_10 = fe_res if fe_success else None
re_res_10 = re_res if re_success else None


In [95]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 102.3963
p-значение: 0.000000
df: 11

Вывод: p < 0.05 - Используйте FIXED EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 2087.4679
p-значение: 0.000000
N (регионов): 56, T (периодов): 77

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -76.0965
p-значение: 1.000000
df: (55, 4301)

Вывод: p >= 0.05 - Pooled адекватна


(102.3963102731729,
 1.1102230246251565e-16,
 2087.4678901751413,
 0.0,
 -76.09648953905308,
 1.0)

In [96]:
###############################
# Построение линейных моделей на панельных данных (кластер 2 ROISFIX)
# (Int_Rate_Mort зависимая переменная)
###############################

exog_vars_initial = [
    'Int_Rate_Mort_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Mort',
   # 'ln_New_Loans_Progr',
    'ln_New_Loans_Mort',
    'ROISFIX',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations'                        
]

dependent_var = 'Int_Rate_Mort'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg_clus_two, df_reg, dependent_var, exog_vars_initial, cov_type='robust', cluster_entity=None, df_name='df_reg_clus_two'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 2 ROISFIX)
Зависимая переменная: Int_Rate_Mort

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:          Int_Rate_Mort   R-squared:                        0.9880
Estimator:                  PooledOLS   R-squared (Between):              0.9997
No. Observations:                 468   R-squared (Within):               0.6431
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9880
Time:                        15:24:46   Log-likelihood                   -590.89
Cov. Estimator:                Robust                                           
                                        F-statistic:                      3413.5
Entities:                           6   P-value                           0.0000
Avg Obs:                       78.000   Distribution:                  F(11,457)
Min Obs:                       78.000                                       

In [97]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_Mort
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
          Credit_impulse  2.302993 0.021728
      Int_Rate_Mort_lag1 -2.259599 0.024317
          Cred_structure -2.082075 0.037893
              Fin_Dostup -2.050433 0.040893
                 ROISFIX  1.934433 0.053678
d_Inflation_Expectations -1.844200 0.065803
             D_top5_rozn  1.725319 0.085147
         Def_Zadolg_Mort  1.669534 0.095698
   Bonds_Rate_Correct_5Y  1.660512 0.097499
               d_Ex_Rate  1.379896 0.168295
               Cred_nagr  1.332226 0.183451

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=93.7339, p=0.000000
White:         stat=245.1776, p=0.000000

Normality tests
Jarque

In [98]:
# Сохраняем результаты модели для экспорта
pooled_res_11 = pooled_res if pooled_success else None
fe_res_11 = fe_res if fe_success else None
re_res_11 = re_res if re_success else None


In [99]:
# ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====
print("\n" + "="*70)
print("ТЕСТЫ СПЕЦИФИКАЦИИ")
print("="*70)


run_spec_tests(
    y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
)



ТЕСТЫ СПЕЦИФИКАЦИИ

ТЕСТЫ СПЕЦИФИКАЦИИ

 ТЕСТ ХАУСМАНА (FE vs RE)
----------------------------------------------------------------------
H-статистика: 4.3690
p-значение: 0.957856
df: 11

Вывод: p >= 0.05 - Используйте RANDOM EFFECTS

 ТЕСТ БРЕУША-ПАГАНА (RE vs Pooled)
----------------------------------------------------------------------
LM статистика: 226.0464
p-значение: 0.000000
N (регионов): 6, T (периодов): 78

Вывод: p < 0.05 - Есть региональные эффекты (используй RE или FE)

 F-ТЕСТ (FE vs Pooled)
----------------------------------------------------------------------
F-статистика: -89.4763
p-значение: 1.000000
df: (5, 451)

Вывод: p >= 0.05 - Pooled адекватна


(4.368976397083882,
 0.957855566360062,
 226.04644539283996,
 0.0,
 -89.47626973222528,
 1.0)

In [100]:
###############################
# Построение линейных моделей на панельных данных (кластер 3 ROISFIX)
# (Int_Rate_Mort зависимая переменная)
###############################

exog_vars_initial = [
    'Int_Rate_Mort_lag1',
    'Cred_nagr',
    'D_top5_rozn',
    'Fin_Dostup',
    'Credit_impulse',
    'Cred_structure',
    'Def_Zadolg_Mort',
   # 'ln_New_Loans_Progr',
    'ln_New_Loans_Mort',
    'ROISFIX',
    'Bonds_Rate_Correct_5Y',
    'd_Ex_Rate',
    'd_Inflation_Expectations'                        
]

dependent_var = 'Int_Rate_Mort'

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg_clus_three, df_reg, dependent_var, exog_vars_initial, cov_type='robust', cluster_entity=None, df_name='df_reg_clus_three'
)


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (КЛАСТЕР 3 ROISFIX)
Зависимая переменная: Int_Rate_Mort

МОДЕЛЬ 1: POOLED OLS
                          PooledOLS Estimation Summary                          
Dep. Variable:          Int_Rate_Mort   R-squared:                        0.9959
Estimator:                  PooledOLS   R-squared (Between):              0.9999
No. Observations:                1482   R-squared (Within):               0.7609
Date:                Tue, Dec 30 2025   R-squared (Overall):              0.9959
Time:                        15:24:46   Log-likelihood                   -1161.9
Cov. Estimator:                Robust                                           
                                        F-statistic:                   3.242e+04
Entities:                          19   P-value                           0.0000
Avg Obs:                       78.000   Distribution:                 F(11,1471)
Min Obs:                       78.000                                       

In [101]:
run_panel_model_diagnostics(
    y, X, pooled_res, fe_res, re_res,
    pooled_success, fe_success, re_success
)



MODEL DIAGNOSTICS
Target: Int_Rate_Mort
p-value threshold: 0.05

Endogeneity (Durbin-Wu-Hausman, control-function)
H0: regressor is exogenous (p < threshold suggests endogeneity).
                variable    t_stat    p_val
      Int_Rate_Mort_lag1 -2.758276 0.005883
         Def_Zadolg_Mort -2.736133 0.006291
   Bonds_Rate_Correct_5Y -2.575598 0.010104
               d_Ex_Rate -2.315977 0.020697
             D_top5_rozn -2.228279 0.026013
               Cred_nagr -2.222979 0.026369
          Credit_impulse  2.182893 0.029201
d_Inflation_Expectations -2.093095 0.036512
                 ROISFIX  2.048771 0.040662
          Cred_structure -1.890607 0.058873
              Fin_Dostup  0.242009 0.808807

----------------------------------------------------------------------
MODEL: Pooled OLS
----------------------------------------------------------------------

Heteroskedasticity tests
Breusch-Pagan: stat=85.2466, p=0.000000
White:         stat=341.9401, p=0.000000

Normality tests
Jarque

In [102]:
# Сохраняем результаты модели для экспорта
pooled_res_12 = pooled_res if pooled_success else None
fe_res_12 = fe_res if fe_success else None
re_res_12 = re_res if re_success else None


In [103]:
'''df_clean.head()

###############################
# Построение PVAR S-GMM моделей
###############################
from pydynpd import regression

# Подготовка данных
df_for_gmm = df_clean.reset_index()
df_for_gmm = df_for_gmm.sort_values(['Region', 'Date'])

# System GMM 

# Используем спецификацию, которая работала ранее
command_str_system = (
    'Int_Rate_FL L1.Int_Rate_FL '
    'Cred_nagr D_top5_rozn '
    'd_Mon_Shock Bonds_Rate_Correct_5Y d_Ex_Rate | '
    'gmm(Int_Rate_FL, 2:4) '
    'iv(Cred_nagr D_top5_rozn d_Mon_Shock Bonds_Rate_Correct_5Y d_Ex_Rate) | '
    'collapse'
)

mydpd_system = regression.abond(command_str_system, df_for_gmm, ['Region', 'Date'])'''


"df_clean.head()\n\n###############################\n# Построение PVAR S-GMM моделей\n###############################\nfrom pydynpd import regression\n\n# Подготовка данных\ndf_for_gmm = df_clean.reset_index()\ndf_for_gmm = df_for_gmm.sort_values(['Region', 'Date'])\n\n# System GMM \n\n# Используем спецификацию, которая работала ранее\ncommand_str_system = (\n    'Int_Rate_FL L1.Int_Rate_FL '\n    'Cred_nagr D_top5_rozn '\n    'd_Mon_Shock Bonds_Rate_Correct_5Y d_Ex_Rate | '\n    'gmm(Int_Rate_FL, 2:4) '\n    'iv(Cred_nagr D_top5_rozn d_Mon_Shock Bonds_Rate_Correct_5Y d_Ex_Rate) | '\n    'collapse'\n)\n\nmydpd_system = regression.abond(command_str_system, df_for_gmm, ['Region', 'Date'])"

In [104]:
from model_results_export import ModelResultsAggregator, ensure_results_dir, add_model_set, build_and_export
import os

dep_var_name = 'Int_Rate_Mort'
base_name = dep_var_name
if base_name.startswith(''):
    base_name = base_name[2:]
if base_name.endswith(''):
    base_name = base_name[:-4]

results_dir = ensure_results_dir('Results')

model_specs_all = [
    {
        'spec_name': 'Модель_1',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_1,
            'fe': fe_res_1,
            're': re_res_1
        }
    },
    {
        'spec_name': 'Модель_2',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_2,
            'fe': fe_res_2,
            're': re_res_2
        }
    },
    {
        'spec_name': 'Модель_3',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_3,
            'fe': fe_res_3,
            're': re_res_3
        }
    },
    {
        'spec_name': 'Модель_4',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_4,
            'fe': fe_res_4,
            're': re_res_4
        }
    },
    {
        'spec_name': 'Модель_5',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_5,
            'fe': fe_res_5,
            're': re_res_5
        }
    },
    {
        'spec_name': 'Модель_6',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_9,
            'fe': fe_res_9,
            're': re_res_9
        }
    }
]

model_specs_cluster = [
    {
        'spec_name': 'Модель_1',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 1',
        'results': {
            'pooled': pooled_res_6,
            'fe': fe_res_6,
            're': re_res_6
        }
    },
    {
        'spec_name': 'Модель_2',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 1',
        'results': {
            'pooled': pooled_res_10,
            'fe': fe_res_10,
            're': re_res_10
        }
    },
    {
        'spec_name': 'Модель_3',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 2',
        'results': {
            'pooled': pooled_res_7,
            'fe': fe_res_7,
            're': re_res_7
        }
    },
    {
        'spec_name': 'Модель_4',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 2',
        'results': {
            'pooled': pooled_res_11,
            'fe': fe_res_11,
            're': re_res_11
        }
    },
    {
        'spec_name': 'Модель_5',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 3',
        'results': {
            'pooled': pooled_res_8,
            'fe': fe_res_8,
            're': re_res_8
        }
    },
    {
        'spec_name': 'Модель_6',
        'dependent_var': dep_var_name,
        'subsample': 'Кластер 3',
        'results': {
            'pooled': pooled_res_12,
            'fe': fe_res_12,
            're': re_res_12
        }
    }
]

aggregator_all = ModelResultsAggregator()
for spec in model_specs_all:
    add_model_set(aggregator_all, spec)

out_all = os.path.join(results_dir, f"{base_name}_all_data.xlsx")
build_and_export(aggregator_all, out_all, include_pvalues=True, decimals=3)

aggregator_cluster = ModelResultsAggregator()
for spec in model_specs_cluster:
    add_model_set(aggregator_cluster, spec)

out_cluster = os.path.join(results_dir, f"{base_name}_cluster_data.xlsx")
build_and_export(aggregator_cluster, out_cluster, include_pvalues=True, decimals=3)


,Модель_1 (POOL),Модель_1 (FE),Модель_1 (RE),Модель_2 (POOL),Модель_2 (FE),Модель_2 (RE),Модель_3 (POOL),Модель_3 (FE),Модель_3 (RE),Модель_4 (POOL),Модель_4 (FE),Модель_4 (RE),Модель_5 (POOL),Модель_5 (FE),Модель_5 (RE),Модель_6 (POOL),Модель_6 (FE),Модель_6 (RE)
Зависимая переменная,Int_Rate_Mort,Int_Rate_Mort,Int_Rate_Mort,Int_Rate_Mort,Int_Rate_Mort,Int_Rate_Mort,Int_Rate_Mort,Int_Rate_Mort,Int_Rate_Mort,Int_Rate_Mort,Int_Rate_Mort,Int_Rate_Mort,Int_Rate_Mort,Int_Rate_Mort,Int_Rate_Mort,Int_Rate_Mort,Int_Rate_Mort,Int_Rate_Mort
Int_Rate_Mort_lag1,0.868*** (0.000),0.803*** (0.000),0.868*** (0.000),0.871*** (0.000),0.798*** (0.000),0.871*** (0.000),0.771*** (0.000),0.673*** (0.000),0.771*** (0.000),0.766*** (0.000),0.668*** (0.000),0.766*** (0.000),0.854*** (0.000),0.806*** (0.000),0.854*** (0.000),0.858*** (0.000),0.774*** (0.000),0.858*** (0.000)
Cred_nagr,0.582*** (0.000),-0.000 (1.000),0.582*** (0.000),0.491*** (0.006),0.116 (0.772),0.491*** (0.006),-0.270 (0.764),1.861 (0.289),-0.270 (0.764),-0.291 (0.734),2.430 (0.165),-0.291 (0.734),0.499** (0.016),-0.061 (0.894),0.499** (0.016),0.555*** (0.008),-0.047 (0.921),0.555*** (0.008)
D_top5_rozn,1.016*** (0.000),-0.365 (0.672),1.016*** (0.000),1.071*** (0.000),-2.360*** (0.005),1.071*** (0.000),2.703* (0.051),-9.909*** (0.009),2.703* (0.051),3.182** (0.024),-6.167 (0.144),3.182** (0.024),0.953*** (0.000),0.977 (0.441),0.953*** (0.000),0.593** (0.021),-0.984 (0.453),0.593** (0.021)
Fin_Dostup,-0.002 (0.401),0.006 (0.353),-0.002 (0.401),0.000 (0.939),0.009 (0.280),0.000 (0.939),0.002 (0.877),-0.028 (0.386),0.002 (0.877),-0.000 (0.973),-0.020 (0.548),-0.000 (0.973),0.009** (0.045),0.005 (0.692),0.009** (0.045),0.015*** (0.004),0.024* (0.062),0.015*** (0.004)
Credit_impulse,0.012*** (0.000),0.013*** (0.000),0.012*** (0.000),0.010*** (0.000),0.012*** (0.000),0.010*** (0.000),0.018** (0.016),0.014* (0.067),0.018** (0.016),0.020** (0.010),0.017** (0.037),0.020** (0.010),0.013*** (0.000),0.016*** (0.000),0.013*** (0.000),0.012*** (0.000),0.018*** (0.000),0.012*** (0.000)
Cred_structure,0.113 (0.413),-0.516 (0.565),0.113 (0.413),-0.245 (0.299),-1.872*** (0.002),-0.245 (0.299),-1.556 (0.231),-2.256 (0.109),-1.556 (0.231),-2.554* (0.092),-4.280** (0.019),-2.554* (0.092),0.204 (0.443),-1.608* (0.090),0.204 (0.443),0.125 (0.642),-2.971*** (0.002),0.125 (0.642)
Def_Zadolg_Mort,0.078*** (0.000),-0.046 (0.252),0.078*** (0.000),0.069*** (0.005),-0.039 (0.482),0.069*** (0.005),0.412 (0.103),1.016** (0.029),0.412 (0.103),0.518** (0.042),1.126** (0.012),0.518** (0.042),0.034 (0.175),-0.057 (0.206),0.034 (0.175),0.048* (0.056),-0.044 (0.323),0.048* (0.056)
d_Mon_Shock,-0.038*** (0.000),-0.039*** (0.000),-0.038*** (0.000),,,,0.009 (0.718),0.006 (0.811),0.009 (0.718),,,,-0.037*** (0.000),-0.038*** (0.000),-0.037*** (0.000),,,
Bonds_Rate_Correct_5Y,-0.057*** (0.001),-0.142*** (0.000),-0.057*** (0.001),-0.014 (0.593),-0.024 (0.452),-0.014 (0.593),-0.142 (0.125),-0.158 (0.122),-0.142 (0.125),-0.086 (0.473),-0.041 (0.775),-0.086 (0.473),-0.051* (0.077),-0.119*** (0.001),-0.051* (0.077),0.004 (0.919),0.015 (0.720),0.004 (0.919)
